# Experiment 3.4: Curvature-Scaled Muon -- Restoring Gradient Magnitude After Orthogonalization

## Scientific Motivation

The Muon optimizer applies Newton-Schulz (NS) orthogonalization to gradient matrices, projecting them
onto the nearest orthogonal matrix. This achieves two things: (1) it aligns the update direction toward
the Newton step (a second-order effect), and (2) it normalizes all singular values to 1, producing a
**unit spectral norm** update.

**The problem:** Experiment 2.20 showed that increasing NS iterations (k=5 to k=20) improves
*directional alignment* toward the true Newton direction -- but paradoxically **degrades final loss**.
Why? Because the Newton step `H^{-1} g` has a natural norm `||H^{-1} g||` that encodes curvature
information -- large steps in flat directions, small steps in steep directions. Orthogonalization
destroys this magnitude information entirely, locking the step to unit spectral norm regardless of
the local curvature landscape.

**Experiment 3.1's failure:** A naive fix of dividing by `sqrt(curvature)` diverged because the
rescaling was unbounded.

## The Fix Tested Here

After orthogonalization, we rescale by a **clamped curvature-aware factor**:

```
scale = clip( ||G||_F / ||ortho_k(G)||_F * gamma,  min=0.1,  max=10.0 )
```

The ratio `||G||_F / ||ortho(G)||_F` recovers the magnitude that orthogonalization stripped away.
The clamp prevents the catastrophic divergence seen in 3.1. The damping factor `gamma` provides
additional control.

We also test an alternative: rescaling by `||velocity||_F` (the momentum buffer norm), which
carries a smoothed history of gradient magnitudes.

## Experimental Design

| Variant | NS iters (k) | Rescaling | Purpose |
|---------|--------------|-----------|---------|
| (a) | k=5 | None | Baseline (standard Muon) |
| (b) | k=20 | None | Confirms k=20 degradation |
| (c) | k=20 | Curvature | Does rescaling fix k=20? |
| (d) | k=20 | Momentum | Alternative magnitude source |
| (e) | k=5 | Curvature | Does rescaling help even at k=5? |

**Key question:** Can we get the directional benefit of high-k orthogonalization while
restoring the magnitude information that makes low-k work better in practice?

In [ ]:
import numpy as np
import os

print("Experiment 3.4: Curvature-Scaled Muon")
print(f"NumPy version: {np.__version__}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## Configuration

The setup mirrors earlier experiments: a 2-layer 4x4 deep linear network (32 total parameters),
trained for 500 steps with learning rate 0.02 and momentum 0.9. This is small enough to run
many seeds quickly, yet rich enough to exhibit the curvature phenomena we care about.

**Key new parameters:**
- `GAMMA = 1.0`: The damping factor in the curvature rescale formula. At gamma=1.0, we recover
  the full pre-orthogonalization magnitude. Values < 1 would dampen toward unit norm; values > 1
  would amplify beyond the original gradient magnitude.
- `SCALE_MIN = 0.1, SCALE_MAX = 10.0`: Clamping bounds that prevent the catastrophic divergence
  seen in Experiment 3.1. The key insight from 3.1 was that unbounded rescaling is fatal; these
  bounds provide a safety net while still allowing a 100x dynamic range.
- `NUM_SEEDS = 10`: Enough for statistical confidence on the direction of effects, though not
  enough for tight confidence intervals.

In [ ]:
DIM = 4
NUM_LAYERS = 2
NUM_STEPS = 500
LR = 0.02
MOMENTUM = 0.9
GAMMA = 1.0          # damping factor for curvature rescaling
SCALE_MIN = 0.1
SCALE_MAX = 10.0
NUM_SEEDS = 10
DATA_POINTS = 32     # number of data vectors

print("=== Experiment Configuration ===")
print(f"  Network: {NUM_LAYERS}-layer deep linear, {DIM}x{DIM} weight matrices")
print(f"  Total parameters: {NUM_LAYERS * DIM * DIM}")
print(f"  Training: {NUM_STEPS} steps, lr={LR}, momentum={MOMENTUM}")
print(f"  Data: {DATA_POINTS} input vectors of dimension {DIM}")
print(f"  Curvature rescale: gamma={GAMMA}, clamp=[{SCALE_MIN}, {SCALE_MAX}]")
print(f"  Dynamic range of rescale: {SCALE_MAX/SCALE_MIN:.0f}x (from {SCALE_MIN} to {SCALE_MAX})")
print(f"  Statistical averaging: {NUM_SEEDS} random seeds")
print()
print("  Note: gamma=1.0 means the rescale factor aims to exactly recover ||G||_F,")
print("  the pre-orthogonalization Frobenius norm. The clamp prevents the factor")
print(f"  from leaving [{SCALE_MIN}, {SCALE_MAX}], avoiding the divergence of Exp 3.1.")

## Network Utilities -- Deep Linear Model

We use a 2-layer deep linear network `Y = W2 @ W1 @ X` as our testbed. Despite computing a
linear function, the **optimization landscape** is highly nonlinear due to the matrix product
structure. This creates a non-trivial loss surface with:

- **Saddle points** arising from the product parameterization
- **Curvature anisotropy** (different singular value directions have different curvatures)
- **Scale ambiguity** (W2 @ W1 = (W2 * c) @ (W1 / c) for any scalar c)

The initialization near identity (`I + 0.1 * noise`) ensures we start in a well-conditioned
region where the gradient signal is clean and the effects of orthogonalization are interpretable.

These properties make deep linear networks the ideal minimal testbed for studying how
orthogonalization interacts with curvature -- the phenomena are real but tractable.

In [ ]:
def init_weights(dim, num_layers, seed):
    """Initialize layers near identity."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(dim) + rng.randn(dim, dim) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
# --- Diagnostic: Inspect initialization properties ---
_demo_weights = init_weights(DIM, NUM_LAYERS, seed=42)
print("=== Initialization Diagnostics (seed=42) ===")
for layer_idx, W in enumerate(_demo_weights):
    svs = np.linalg.svd(W, compute_uv=False)
    cond = svs[0] / svs[-1] if svs[-1] > 1e-12 else float('inf')
    print(f"  Layer {layer_idx}: shape={W.shape}, ||W||_F={np.linalg.norm(W, 'fro'):.4f}, "
          f"sigma_range=[{svs[-1]:.4f}, {svs[0]:.4f}], kappa={cond:.4f}")
print(f"  Product ||W2 @ W1||_F = {np.linalg.norm(_demo_weights[1] @ _demo_weights[0], 'fro'):.4f}")
print(f"  Product condition number = {np.linalg.cond(_demo_weights[1] @ _demo_weights[0]):.4f}")
print()
print("  Interpretation: Near-identity initialization gives condition numbers close to 1,")
print("  meaning the initial curvature landscape is relatively isotropic. As training proceeds,")
print("  the weights will develop anisotropy, making the curvature rescaling increasingly relevant.")
del _demo_weights

In [ ]:
def forward_linear(weights, X):
    """Forward pass through deep linear net."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y_target):
    """MSE loss."""
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(diff ** 2)

In [ ]:
def compute_gradients(weights, X, Y_target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    batch_size = X.shape[1]

    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])

    delta = (activations[-1] - Y_target) / batch_size

    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta

    return grads

## Newton-Schulz Orthogonalization -- The Core of Muon

The Newton-Schulz iteration is an **iterative polar decomposition** that converges to the nearest
orthogonal matrix `U` in the polar decomposition `G = U * S`. Muon uses a quintic variant with
coefficients `a=3.4445, b=-4.7750, c=2.0315` (optimized for fast convergence):

```
X_{k+1} = a*X + b*X@(X^T@X) + c*X@(X^T@X)^2
```

**What it does to singular values:** If G has SVD `G = U diag(s1, s2, ...) V^T`, then after
sufficient iterations, all singular values converge to 1: `ortho(G) ~ U * I * V^T`. This is
the orthogonal Procrustes solution.

**What this experiment tests:** The convergence rate depends on k (number of iterations).
At k=5, the singular values are *approximately* equalized but retain some residual spread.
At k=20, they are *exactly* equalized. The residual spread at k=5 acts as an **implicit
curvature signal** -- singular values corresponding to high-curvature directions naturally
get compressed more. By equalizing perfectly at k=20, we destroy this implicit signal.

The curvature rescaling aims to **explicitly restore** what k=20 implicitly destroys.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=5):
    """Newton-Schulz iteration with Muon's quintic coefficients.
    a=3.4445, b=-4.7750, c=2.0315
    X_{k+1} = a*X + b*X@(X^T@X) + c*X@(X^T@X)^2
    """
    a, b, c = 3.4445, -4.7750, 2.0315
    norm = np.linalg.norm(G, 'fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        XtX = X.T @ X
        X_XtX = X @ XtX
        XtX2 = XtX @ XtX
        X_XtX2 = X @ XtX2
        X = a * X + b * X_XtX + c * X_XtX2

    return X

In [ ]:
# --- Diagnostic: NS orthogonalization at k=5 vs k=20 ---
# Show how singular values converge and what the norm ratio looks like
print("=== Newton-Schulz Convergence Diagnostic ===")
print()
_rng = np.random.RandomState(99)
_G_demo = _rng.randn(DIM, DIM) * 0.5 + np.eye(DIM) * 0.1  # Typical gradient-like matrix
_svs_orig = np.linalg.svd(_G_demo, compute_uv=False)
_G_norm = np.linalg.norm(_G_demo, 'fro')
print(f"  Original gradient G:")
print(f"    Singular values: {np.array2string(_svs_orig, precision=4)}")
print(f"    ||G||_F = {_G_norm:.4f}")
print(f"    Condition number = {_svs_orig[0]/_svs_orig[-1]:.4f}")
print()

for _k in [5, 10, 20]:
    _G_orth = newton_schulz_orthogonalize(_G_demo, num_iters=_k)
    _svs_orth = np.linalg.svd(_G_orth, compute_uv=False)
    _orth_norm = np.linalg.norm(_G_orth, 'fro')
    _scale_ratio = _G_norm / _orth_norm if _orth_norm > 1e-12 else float('inf')
    _sv_spread = np.max(_svs_orth) - np.min(_svs_orth)
    print(f"  After k={_k} NS iterations:")
    print(f"    Singular values: {np.array2string(_svs_orth, precision=6)}")
    print(f"    SV spread (max-min): {_sv_spread:.8f}  {'(near-perfect ortho)' if _sv_spread < 1e-4 else '(residual spread)'}")
    print(f"    ||ortho(G)||_F = {_orth_norm:.4f}")
    print(f"    Curvature rescale factor = ||G||/||ortho(G)|| = {_scale_ratio:.4f}")
    print()

print("  Key insight: The rescale factor ||G||/||ortho(G)|| captures how much")
print("  magnitude information was lost. At k=5 some residual SV spread remains;")
print("  at k=20 the SVs are exactly 1.0 so the norm is sqrt(min(m,n)) = sqrt(4) = 2.0.")
del _rng, _G_demo, _svs_orig, _G_norm, _G_orth, _svs_orth, _orth_norm, _scale_ratio, _sv_spread, _k

## Optimizer: Muon with Optional Curvature Rescaling

The training loop implements three modes of operation:

### Mode 1: Standard Muon (`rescale_mode='none'`)
The vanilla Muon pipeline: compute gradient G, orthogonalize via NS iteration, apply momentum,
update weights. The orthogonalized update has unit spectral norm by construction.

### Mode 2: Curvature Rescaling (`rescale_mode='curvature'`)
After orthogonalization, multiply by `scale = clip(||G||_F / ||ortho(G)||_F * gamma, min, max)`.

**Physical interpretation:** The ratio `||G||_F / ||ortho(G)||_F` measures how much the
Frobenius norm changed during orthogonalization. For a gradient with large singular value
spread (high curvature anisotropy), this ratio can be large -- orthogonalization compressed
the dominant singular values significantly. By rescaling, we restore the *total magnitude*
while keeping the *direction* from the orthogonalized update.

This is a scalar rescaling -- it cannot restore per-singular-value magnitude differences.
It is a compromise between the full curvature information (which diverges unbounded, per 3.1)
and no information at all (standard Muon).

### Mode 3: Momentum Rescaling (`rescale_mode='momentum'`)
After orthogonalization, multiply by `scale = clip(||velocity||_F, min, max)`.

**Physical interpretation:** The momentum buffer accumulates past orthogonalized gradients.
Its norm reflects the *sustained* gradient magnitude over recent history. Using this as a
scale factor provides a temporally smoothed estimate of the appropriate step size, independent
of the curvature rescaling mechanism.

In [ ]:
def train_muon(weights, X, Y_target, lr, num_steps, ns_iters=5,
               rescale_mode='none', gamma=1.0, scale_min=0.1, scale_max=10.0,
               momentum=0.9):
    """
    Train with Muon optimizer.

    rescale_mode:
      'none'      -- standard Muon (no rescaling after ortho)
      'curvature' -- scale = clip(||G||_F / ||ortho(G)||_F * gamma, min, max)
      'momentum'  -- scale = ||velocity||_F  (gradient magnitude info)
    """
    num_layers = len(weights)
    velocities = [np.zeros_like(W) for W in weights]
    losses = []
    scales_used = []  # track per-step average scale across layers

    for step in range(num_steps):
        loss = compute_loss(weights, X, Y_target)
        losses.append(loss)

        grads = compute_gradients(weights, X, Y_target)

        step_scales = []
        for i in range(num_layers):
            G = grads[i]
            G_norm = np.linalg.norm(G, 'fro')

            # Newton-Schulz orthogonalization
            G_orth = newton_schulz_orthogonalize(G, num_iters=ns_iters)
            G_orth_norm = np.linalg.norm(G_orth, 'fro')

            # Compute rescaling factor
            if rescale_mode == 'curvature':
                if G_orth_norm > 1e-12:
                    scale = np.clip(G_norm / G_orth_norm * gamma, scale_min, scale_max)
                else:
                    scale = 1.0
                G_orth = G_orth * scale
            elif rescale_mode == 'momentum':
                # Use momentum norm as scale (after update)
                vel_norm = np.linalg.norm(velocities[i], 'fro')
                if vel_norm > 1e-12 and step > 0:
                    scale = np.clip(vel_norm, scale_min, scale_max)
                else:
                    scale = 1.0
                G_orth = G_orth * scale
            else:
                scale = 1.0

            step_scales.append(scale)

            # Momentum update
            velocities[i] = momentum * velocities[i] + G_orth
            weights[i] = weights[i] - lr * velocities[i]

        scales_used.append(np.mean(step_scales))

    final_loss = compute_loss(weights, X, Y_target)
    losses.append(final_loss)

    return losses, scales_used, weights

## Main Experiment -- Five-Way Comparison Across Seeds

We now execute the core experiment: running all 5 variants with identical initialization and
data for each of 10 random seeds. Each seed generates:

1. A random target function `W_target` (2 layers of 4x4 matrices, scaled by 0.3)
2. Random input data `X` (4x32 matrix, scaled by 0.5)
3. Target outputs `Y = W2_target @ W1_target @ X`
4. Identical initial weights for all 5 variants (using `seed + 1000`)

The seed offset for initialization (`+1000`) ensures that the initial weights are decorrelated
from the target, preventing any accidental alignment artifacts.

**What to watch for in the per-seed results:**
- Does variant (b) consistently lose to (a)? This confirms the k=20 degradation.
- Does variant (c) rescue (b)'s performance? This validates curvature rescaling.
- How does variant (d) compare? Momentum rescaling is a fundamentally different signal source.
- Is variant (e) ever better than (a)? This would suggest even k=5 benefits from explicit magnitude.

In [ ]:
def run_single_seed(seed):
    """Run all 5 variants for a single seed and return results."""
    rng = np.random.RandomState(seed)

    # Generate random target
    W_target = [rng.randn(DIM, DIM) * 0.3 for _ in range(NUM_LAYERS)]
    X = rng.randn(DIM, DATA_POINTS) * 0.5
    Y_target = X.copy()
    for W in W_target:
        Y_target = W @ Y_target

    variants = {
        '(a) Muon k=5': dict(ns_iters=5, rescale_mode='none'),
        '(b) Muon k=20': dict(ns_iters=20, rescale_mode='none'),
        '(c) k=20 + curv rescale': dict(ns_iters=20, rescale_mode='curvature', gamma=GAMMA),
        '(d) k=20 + mom rescale': dict(ns_iters=20, rescale_mode='momentum'),
        '(e) k=5 + curv rescale': dict(ns_iters=5, rescale_mode='curvature', gamma=GAMMA),
    }

    results = {}
    for name, kwargs in variants.items():
        w_init = init_weights(DIM, NUM_LAYERS, seed + 1000)
        losses, scales, _ = train_muon(
            w_init, X, Y_target, lr=LR, num_steps=NUM_STEPS,
            momentum=MOMENTUM, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
            **kwargs
        )
        results[name] = {
            'losses': losses,
            'scales': scales,
            'final_loss': losses[-1],
        }

    return results

In [ ]:
# --- Diagnostic: Verify single-seed mechanics before full run ---
print("=== Single-Seed Dry Run (seed=42) ===")
print()
_demo_result = run_single_seed(42)
for _name, _res in _demo_result.items():
    _init_loss = _res['losses'][0]
    _final_loss = _res['final_loss']
    _reduction = (_init_loss - _final_loss) / _init_loss * 100 if _init_loss > 1e-15 else 0
    _scale_info = ""
    if 'rescale' in _name:
        _scales = _res['scales']
        _scale_info = f", scale_range=[{min(_scales):.3f}, {max(_scales):.3f}], scale_mean={np.mean(_scales):.3f}"
    print(f"  {_name}:")
    print(f"    Initial loss: {_init_loss:.6e}")
    print(f"    Final loss:   {_final_loss:.6e}  ({_reduction:.1f}% reduction)")
    if _scale_info:
        print(f"    Rescale stats: {_scale_info}")
print()
print("  Sanity check: All variants start from the same loss (same init weights).")
print(f"  Initial losses match: {len(set(round(_demo_result[n]['losses'][0], 10) for n in _demo_result)) == 1}")
del _demo_result, _name, _res, _init_loss, _final_loss, _reduction, _scale_info, _scales

### Interpreting the Dry Run

The dry run above serves multiple purposes:

1. **Initialization consistency:** All 5 variants must begin from the same initial loss, confirming
   they share identical starting weights and differ only in the optimizer configuration.

2. **Rescale factor range:** The curvature rescale factors for variants (c) and (e) should be
   bounded within [0.1, 10.0]. If many values hit the clamp boundaries, the clamp range may be
   too restrictive. If they cluster near 1.0, orthogonalization is not significantly changing
   the norm, and rescaling may have little effect.

3. **Momentum rescale dynamics:** Variant (d) uses `||velocity||_F` as the scale. Since the
   velocity starts at zero and builds up via momentum accumulation, early steps should show
   scale = 1.0 (the fallback). The scale should grow as the velocity buffer accumulates,
   potentially creating a learning rate warmup effect.

## Full Multi-Seed Experiment Execution

In [ ]:
def main():
    print()
    print("=" * 100)
    print("  Experiment 3.4: Curvature-Scaled Muon")
    print("=" * 100)
    print()
    print("  CONTEXT: k=20 NS gives better Newton alignment but HURTS loss because step")
    print("  size is locked at unit spectral norm. Fix: rescale by clamped curvature factor.")
    print()
    print(f"  Config: {NUM_LAYERS}-layer {DIM}x{DIM} deep linear, {NUM_STEPS} steps, lr={LR}")
    print(f"  Curvature rescale: scale = clip(||G||/||ortho(G)|| * gamma, {SCALE_MIN}, {SCALE_MAX})")
    print(f"  gamma = {GAMMA}, momentum = {MOMENTUM}")
    print(f"  Averaging over {NUM_SEEDS} seeds")
    print()

    # =========================================================================
    # Run all seeds
    # =========================================================================
    variant_names = [
        '(a) Muon k=5',
        '(b) Muon k=20',
        '(c) k=20 + curv rescale',
        '(d) k=20 + mom rescale',
        '(e) k=5 + curv rescale',
    ]

    all_final_losses = {name: [] for name in variant_names}
    all_loss_curves = {name: [] for name in variant_names}
    all_scale_curves = {name: [] for name in variant_names}

    print("  --- Running seeds ---")
    for i in range(NUM_SEEDS):
        seed = 42 + i * 137
        results = run_single_seed(seed)
        for name in variant_names:
            all_final_losses[name].append(results[name]['final_loss'])
            all_loss_curves[name].append(results[name]['losses'])
            all_scale_curves[name].append(results[name]['scales'])
        print(f"  Seed {i+1:2d}/{NUM_SEEDS}: "
              + "  ".join(f"{name.split(')')[0]})={results[name]['final_loss']:.2e}"
                         for name in variant_names))

    print()
    print("  All seeds completed. Analyzing results...")
    print()

    # =========================================================================
    # Quick sanity: initial loss consistency across variants
    # =========================================================================
    print("=" * 100)
    print("  SANITY CHECK: Initial loss consistency")
    print("=" * 100)
    print()
    for i in range(min(3, NUM_SEEDS)):
        init_losses = {name: all_loss_curves[name][i][0] for name in variant_names}
        init_vals = list(init_losses.values())
        all_same = all(abs(v - init_vals[0]) < 1e-10 for v in init_vals)
        print(f"  Seed {i+1}: initial_loss = {init_vals[0]:.6e}, all variants identical: {all_same}")
    print()
    print("  (All variants share identical initial weights per seed, so initial losses must match.)")
    print()

    # =========================================================================
    # Early convergence check: where do variants diverge?
    # =========================================================================
    print("=" * 100)
    print("  EARLY CONVERGENCE: When do the variants start to differ?")
    print("=" * 100)
    print()
    for _check_step in [10, 25, 50]:
        print(f"  At step {_check_step} (averaged over seeds):")
        for name in variant_names:
            curves = np.array(all_loss_curves[name])
            mean_at_step = np.mean(curves[:, _check_step])
            short = name.split(')')[0] + ')'
            print(f"    {short:>8}: loss = {mean_at_step:.6e}")
        print()
    print("  Interpretation: If rescaled variants diverge early, the rescaling effect")
    print("  is immediate (magnitude matters from step 1). If they diverge late, the")
    print("  curvature anisotropy needs time to develop.")
    print()

    # =========================================================================
    # Summary Table
    # =========================================================================
    print()
    print("=" * 100)
    print("  SUMMARY TABLE (final loss, mean +/- std over seeds)")
    print("=" * 100)
    print()

    ref_losses = np.array(all_final_losses['(a) Muon k=5'])
    ref_mean = np.mean(ref_losses)

    print(f"  {'Variant':<30s} {'Final loss (mean)':>18s} {'Std':>12s} "
          f"{'vs (a) k=5':>14s} {'Median':>14s}")
    print(f"  {'-'*30} {'-'*18} {'-'*12} {'-'*14} {'-'*14}")

    for name in variant_names:
        fl = np.array(all_final_losses[name])
        fl_mean = np.mean(fl)
        fl_std = np.std(fl)
        fl_median = np.median(fl)
        if ref_mean > 1e-15:
            vs_ref = (fl_mean - ref_mean) / ref_mean * 100
            vs_str = f"{vs_ref:+.1f}%"
        else:
            vs_str = "N/A"
        print(f"  {name:<30s} {fl_mean:18.6e} {fl_std:12.2e} {vs_str:>14s} {fl_median:14.6e}")

    print()
    print("  Reading the table:")
    print("  - Negative '% vs (a)' means BETTER than baseline Muon k=5")
    print("  - Positive '% vs (a)' means WORSE than baseline")
    print("  - Large std relative to mean indicates high seed sensitivity")

    # =========================================================================
    # Rescaling factor analysis
    # =========================================================================
    print()
    print("=" * 100)
    print("  RESCALING FACTOR ANALYSIS")
    print("=" * 100)
    print()
    print("  This section examines the rescaling factors actually used during training.")
    print("  If factors cluster near 1.0, orthogonalization preserves norm well and rescaling")
    print("  has minimal effect. If they deviate significantly, there is substantial magnitude")
    print("  information being lost/restored.")
    print()

    for name in variant_names:
        if 'rescale' in name:
            all_scales = np.array(all_scale_curves[name])  # (num_seeds, num_steps)
            scale_mean = np.mean(all_scales, axis=0)
            print(f"  {name}:")
            print(f"    Scale at step 0:   mean={scale_mean[0]:.4f}")
            print(f"    Scale at step 100: mean={scale_mean[min(100, len(scale_mean)-1)]:.4f}")
            print(f"    Scale at step 250: mean={scale_mean[min(250, len(scale_mean)-1)]:.4f}")
            print(f"    Scale at step 499: mean={scale_mean[-1]:.4f}")
            print(f"    Overall range: [{np.min(all_scales):.4f}, {np.max(all_scales):.4f}]")
            # How often does it hit the clamp?
            hit_min = np.mean(all_scales <= SCALE_MIN + 1e-8) * 100
            hit_max = np.mean(all_scales >= SCALE_MAX - 1e-8) * 100
            print(f"    Fraction hitting min clamp ({SCALE_MIN}): {hit_min:.1f}%")
            print(f"    Fraction hitting max clamp ({SCALE_MAX}): {hit_max:.1f}%")
            # Trend analysis
            _early = np.mean(all_scales[:, :50])
            _late = np.mean(all_scales[:, -50:])
            _trend = "increasing" if _late > _early * 1.05 else ("decreasing" if _late < _early * 0.95 else "stable")
            print(f"    Temporal trend: early_mean={_early:.4f}, late_mean={_late:.4f} -> {_trend}")
            print()

    print("  Interpretation: If curvature rescale factors are far from 1.0, it means")
    print("  orthogonalization is significantly changing the gradient magnitude, and")
    print("  rescaling is doing real work. If they hit the clamp frequently, the clamp")
    print("  bounds may need adjustment.")

    # =========================================================================
    # Loss curve comparison at key steps
    # =========================================================================
    print()
    print("=" * 100)
    print("  LOSS CURVES AT KEY STEPS (averaged over seeds)")
    print("=" * 100)
    print()

    check_steps = [0, 50, 100, 200, 300, 400, 500]
    header = f"  {'Step':>6}"
    for name in variant_names:
        short = name.split(')')[0] + ')'
        header += f"  {short:>16}"
    print(header)
    print(f"  {'-'*6}" + f"  {'-'*16}" * len(variant_names))

    for step in check_steps:
        row = f"  {step:>6}"
        for name in variant_names:
            curves = np.array(all_loss_curves[name])
            if step < curves.shape[1]:
                mean_loss = np.mean(curves[:, step])
                row += f"  {mean_loss:16.6e}"
            else:
                row += f"  {'---':>16}"
        print(row)

    print()
    print("  Interpretation: Look for where the curves cross. If (c) starts above (a) but")
    print("  ends below, the curvature rescaling has a slow-start-fast-finish pattern,")
    print("  suggesting it becomes more beneficial as curvature develops during training.")

    # =========================================================================
    # Convergence speed comparison
    # =========================================================================
    print()
    print("=" * 100)
    print("  CONVERGENCE SPEED: Steps to reach 10x loss reduction")
    print("=" * 100)
    print()

    for name in variant_names:
        curves = np.array(all_loss_curves[name])
        steps_to_10x = []
        for seed_idx in range(NUM_SEEDS):
            curve = curves[seed_idx]
            init_loss = curve[0]
            target = init_loss / 10.0
            reached = np.where(curve <= target)[0]
            if len(reached) > 0:
                steps_to_10x.append(reached[0])
            else:
                steps_to_10x.append(NUM_STEPS)  # never reached
        short = name.split(')')[0] + ')'
        mean_steps = np.mean(steps_to_10x)
        median_steps = np.median(steps_to_10x)
        print(f"  {short:>8}: mean={mean_steps:.0f} steps, median={median_steps:.0f} steps")

    print()
    print("  Faster convergence to 10x reduction = more efficient early optimization.")

    # =========================================================================
    # HYPOTHESIS TESTS
    # =========================================================================
    print()
    print("=" * 100)
    print("  HYPOTHESIS TESTS")
    print("=" * 100)
    print()

    # Gather arrays
    losses_a = np.array(all_final_losses['(a) Muon k=5'])
    losses_b = np.array(all_final_losses['(b) Muon k=20'])
    losses_c = np.array(all_final_losses['(c) k=20 + curv rescale'])
    losses_d = np.array(all_final_losses['(d) k=20 + mom rescale'])
    losses_e = np.array(all_final_losses['(e) k=5 + curv rescale'])

    # Test 1: k=20 is worse than k=5 (confirming the problem)
    mean_a = np.mean(losses_a)
    mean_b = np.mean(losses_b)
    t1_pass = mean_b > mean_a
    pct_1 = (mean_b - mean_a) / mean_a * 100 if mean_a > 1e-15 else float('nan')
    print(f"  T1: k=20 is worse than k=5 (confirms the problem)")
    print(f"      mean(b)={mean_b:.6e} vs mean(a)={mean_a:.6e} ({pct_1:+.1f}%)")
    print(f"      Per-seed wins for k=5: {np.sum(losses_a < losses_b)}/{NUM_SEEDS}")
    print(f"      {'PASS' if t1_pass else 'FAIL'}: k=20 {'IS' if t1_pass else 'is NOT'} worse")
    print()

    # Test 2: Curvature rescaling fixes k=20 (c better than b)
    mean_c = np.mean(losses_c)
    t2_pass = mean_c < mean_b
    pct_2 = (mean_c - mean_b) / mean_b * 100 if mean_b > 1e-15 else float('nan')
    print(f"  T2: Curvature rescaling fixes k=20 degradation? (c < b)")
    print(f"      mean(c)={mean_c:.6e} vs mean(b)={mean_b:.6e} ({pct_2:+.1f}%)")
    print(f"      Per-seed wins for (c): {np.sum(losses_c < losses_b)}/{NUM_SEEDS}")
    print(f"      {'PASS' if t2_pass else 'FAIL'}: rescaled k=20 {'IS' if t2_pass else 'is NOT'} better than plain k=20")
    print()

    # Test 3: Momentum rescaling fixes k=20 (d better than b)
    mean_d = np.mean(losses_d)
    t3_pass = mean_d < mean_b
    pct_3 = (mean_d - mean_b) / mean_b * 100 if mean_b > 1e-15 else float('nan')
    print(f"  T3: Momentum rescaling fixes k=20 degradation? (d < b)")
    print(f"      mean(d)={mean_d:.6e} vs mean(b)={mean_b:.6e} ({pct_3:+.1f}%)")
    print(f"      Per-seed wins for (d): {np.sum(losses_d < losses_b)}/{NUM_SEEDS}")
    print(f"      {'PASS' if t3_pass else 'FAIL'}: mom-rescaled k=20 {'IS' if t3_pass else 'is NOT'} better than plain k=20")
    print()

    # Test 4: Does curvature rescaling make k=20 match or beat k=5? (c <= a)
    t4_pass = mean_c <= mean_a * 1.05  # within 5%
    pct_4 = (mean_c - mean_a) / mean_a * 100 if mean_a > 1e-15 else float('nan')
    print(f"  T4: Does curvature rescaling make k=20 match k=5? (c <= a within 5%)")
    print(f"      mean(c)={mean_c:.6e} vs mean(a)={mean_a:.6e} ({pct_4:+.1f}%)")
    print(f"      Per-seed wins for (c) over (a): {np.sum(losses_c < losses_a)}/{NUM_SEEDS}")
    print(f"      {'PASS' if t4_pass else 'FAIL'}: rescaled k=20 {'MATCHES' if t4_pass else 'does NOT match'} k=5")
    print()

    # Test 5: Does rescaling at k=5 help? (e < a)
    mean_e = np.mean(losses_e)
    t5_pass = mean_e < mean_a
    pct_5 = (mean_e - mean_a) / mean_a * 100 if mean_a > 1e-15 else float('nan')
    print(f"  T5: Does rescaling at k=5 improve further? (e < a)")
    print(f"      mean(e)={mean_e:.6e} vs mean(a)={mean_a:.6e} ({pct_5:+.1f}%)")
    print(f"      Per-seed wins for (e) over (a): {np.sum(losses_e < losses_a)}/{NUM_SEEDS}")
    print(f"      {'PASS' if t5_pass else 'FAIL'}: k=5+rescale {'IS' if t5_pass else 'is NOT'} better than plain k=5")
    print()

    # Test 6: Best variant overall
    variant_means = {
        '(a)': mean_a, '(b)': mean_b, '(c)': mean_c,
        '(d)': mean_d, '(e)': mean_e,
    }
    best_name = min(variant_means, key=variant_means.get)
    best_val = variant_means[best_name]
    print(f"  T6: Best variant overall: {best_name} with mean final loss = {best_val:.6e}")
    print()

    # =========================================================================
    # Effect size analysis
    # =========================================================================
    print("=" * 100)
    print("  EFFECT SIZE ANALYSIS (Cohen's d)")
    print("=" * 100)
    print()
    print("  Cohen's d measures the standardized difference between two groups.")
    print("  |d| < 0.2: negligible, 0.2-0.5: small, 0.5-0.8: medium, > 0.8: large")
    print()

    _pairs = [
        ("(a) vs (b): k=5 vs k=20 gap", losses_a, losses_b),
        ("(c) vs (b): curvature rescale improvement", losses_c, losses_b),
        ("(d) vs (b): momentum rescale improvement", losses_d, losses_b),
        ("(c) vs (a): curvature rescale vs baseline", losses_c, losses_a),
        ("(e) vs (a): k=5+rescale vs k=5 baseline", losses_e, losses_a),
    ]
    for _label, _x, _y in _pairs:
        _pooled_std = np.sqrt((np.var(_x) + np.var(_y)) / 2)
        if _pooled_std > 1e-15:
            _d = (np.mean(_x) - np.mean(_y)) / _pooled_std
            _size = "negligible" if abs(_d) < 0.2 else ("small" if abs(_d) < 0.5 else ("medium" if abs(_d) < 0.8 else "large"))
            print(f"  {_label}: d = {_d:+.3f} ({_size})")
        else:
            print(f"  {_label}: d = N/A (zero variance)")

    # =========================================================================
    # Per-seed detail table
    # =========================================================================
    print()
    print("=" * 100)
    print("  PER-SEED FINAL LOSSES")
    print("=" * 100)
    print()
    header = f"  {'Seed':>6}"
    for name in variant_names:
        short = name.split(')')[0] + ')'
        header += f"  {short:>16}"
    header += f"  {'Best':>8}"
    print(header)
    print(f"  {'-'*6}" + f"  {'-'*16}" * len(variant_names) + f"  {'-'*8}")

    for i in range(NUM_SEEDS):
        row = f"  {i+1:>6}"
        seed_losses = {}
        for j, name in enumerate(variant_names):
            fl = all_final_losses[name][i]
            row += f"  {fl:16.6e}"
            seed_losses[name.split(')')[0] + ')'] = fl
        best = min(seed_losses, key=seed_losses.get)
        row += f"  {best:>8}"
        print(row)

    # Win count summary
    print()
    print("  Win counts (which variant has lowest final loss per seed):")
    _win_counts = {name.split(')')[0] + ')': 0 for name in variant_names}
    for i in range(NUM_SEEDS):
        seed_losses = {name.split(')')[0] + ')': all_final_losses[name][i] for name in variant_names}
        best = min(seed_losses, key=seed_losses.get)
        _win_counts[best] += 1
    for _k, _v in _win_counts.items():
        print(f"    {_k}: {_v}/{NUM_SEEDS} wins")

    # =========================================================================
    # OVERALL VERDICT
    # =========================================================================
    print()
    print("=" * 100)
    print("  OVERALL VERDICT")
    print("=" * 100)
    print()

    if t1_pass:
        print("  [CONFIRMED] k=20 is worse than k=5 (the known problem).")
    else:
        print("  [UNEXPECTED] k=20 is NOT worse than k=5 in this run.")

    if t2_pass and t4_pass:
        print("  [SUCCESS] Curvature rescaling FIXES k=20 degradation and matches/beats k=5.")
    elif t2_pass:
        print("  [PARTIAL] Curvature rescaling improves k=20 but does not fully match k=5.")
    else:
        print("  [FAIL] Curvature rescaling does NOT fix k=20.")

    if t3_pass:
        print("  [SUCCESS] Momentum rescaling also fixes k=20 degradation.")
    else:
        print("  [FAIL] Momentum rescaling does NOT fix k=20.")

    if t5_pass:
        print("  [BONUS] Rescaling at k=5 provides further improvement.")
    else:
        print("  [NO BONUS] Rescaling at k=5 does not help (k=5 already good enough).")

    print()
    print("=" * 100)
    print("  EXPERIMENT COMPLETE")
    print("=" * 100)

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions and Scientific Interpretation

### What This Experiment Tests

The fundamental question is whether the **magnitude information** destroyed by Newton-Schulz
orthogonalization is a significant loss. Standard Muon at k=5 retains some residual singular
value spread (an implicit magnitude signal), while k=20 eliminates it entirely. This experiment
tests whether explicitly restoring magnitude via a clamped rescaling factor can recover or
surpass k=5 performance while keeping k=20's superior directional alignment.

### The Rescaling Mechanism in Detail

The curvature rescale factor `||G||_F / ||ortho(G)||_F` has a clean geometric interpretation:

- For a perfectly orthogonal gradient (all SVs equal), this ratio equals 1 -- no information was lost.
- For a highly anisotropic gradient (one dominant SV), the orthogonalized version has `||ortho||_F = sqrt(n)` while `||G||_F` can be much larger, giving a scale >> 1. This means orthogonalization compressed the gradient significantly, and the rescaling restores the lost magnitude.

The clamping to [0.1, 10.0] is the critical difference from Experiment 3.1, which diverged with unbounded rescaling. The 100x dynamic range allows meaningful magnitude restoration while preventing catastrophic step sizes.

### Implications for Muon Design

The results of this experiment have direct implications for practical Muon implementations:

1. **If curvature rescaling helps at k=20 (T2 PASS):** The magnitude information is genuinely
   important, and orthogonalization is discarding a useful signal. This motivates exploring
   per-singular-value rescaling (not just scalar) in future work.

2. **If curvature rescaling makes k=20 match k=5 (T4 PASS):** The k=5 advantage was entirely
   due to its implicit magnitude retention, not due to k=5 finding a better direction. This
   confirms that direction and magnitude are separable concerns in Muon.

3. **If momentum rescaling also works (T3 PASS):** The magnitude signal does not need to come
   from the current gradient -- historical information suffices. This opens the door to
   adaptive step sizing schemes for Muon.

4. **If k=5 + rescaling beats plain k=5 (T5 PASS):** Even the standard Muon configuration is
   suboptimal in its magnitude handling, suggesting room for improvement in the production optimizer.

### Connection to the RG Gauge-Fixing Framework

In the renormalization group (RG) interpretation of Muon, orthogonalization acts as a gauge
fixing that removes redundant scale degrees of freedom. The curvature rescaling tested here
can be understood as **restoring the physical scale** after gauge fixing -- analogous to how
RG-improved perturbation theory must carefully handle the running of coupling constants
after fixing the renormalization scale. The clamping bounds correspond to the requirement
that the running coupling remain in the perturbative regime.

### Next Steps

- **Per-SV rescaling:** Instead of scalar `||G||/||ortho(G)||`, restore individual singular
  values from the pre-orthogonalization gradient. This would give a richer magnitude signal.
- **Adaptive clamping:** Learn or schedule the clamp bounds during training.
- **Scale to real networks:** Test on CNNs/transformers where curvature anisotropy is extreme.